# TrueLayer — HSBC Data Exploration

Uses the TrueLayer Data API (Open Banking) to fetch account and transaction data from HSBC.

Requires in `.env`:
- `TRUELAYER_CLIENT_ID`
- `TRUELAYER_CLIENT_SECRET`

**Flow:**
1. Run the **Setup** cell
2. Run the **Authorise** cell — it opens the browser and waits automatically
3. Query accounts, balances, and transactions

Set `SANDBOX = True` to use TrueLayer's mock provider (no real bank login needed).

In [ ]:
import httpx
from dotenv import dotenv_values

env = dotenv_values("../.env")

CLIENT_ID     = env["TRUELAYER_CLIENT_ID"]
CLIENT_SECRET = env["TRUELAYER_CLIENT_SECRET"]
REDIRECT_URI  = "https://console.truelayer.com/redirect-page"
SANDBOX       = False  # set False to hit the real HSBC provider

AUTH_BASE = "https://auth.truelayer.com"  # always — sandbox is determined by the sandbox- client_id prefix
API_BASE  = "https://api.truelayer-sandbox.com" if SANDBOX else "https://api.truelayer.com"

print(f"Mode:     {'sandbox' if SANDBOX else 'production'}")
print(f"ClientID: {CLIENT_ID}")


In [ ]:
# Debug — verify credentials are loaded cleanly (no quotes/spaces)
print(f"CLIENT_ID:     '{CLIENT_ID}'")
print(f"CLIENT_SECRET: '{CLIENT_SECRET[:4]}...{CLIENT_SECRET[-4:]}'  len={len(CLIENT_SECRET)}")
print(f"REDIRECT_URI:  '{REDIRECT_URI}'")

## Authorise

Run this cell to open the auth URL. Log in, then copy the `code` value from the redirect URL and paste it into the next cell.

In [ ]:
import urllib.parse
import webbrowser

SCOPES    = "info accounts balance transactions offline_access"
PROVIDERS = "mock" if SANDBOX else "hsbc"

auth_url = (
    f"{AUTH_BASE}/?"
    + urllib.parse.urlencode({
        "response_type": "code",
        "client_id":     CLIENT_ID,
        "scope":         SCOPES,
        "redirect_uri":  REDIRECT_URI,
        "providers":     PROVIDERS,
    })
)

print(auth_url)
webbrowser.open(auth_url)

In [ ]:
AUTH_CODE = ""  # re-run the auth cell above to get a new code

resp = httpx.post(
    f"{AUTH_BASE}/connect/token",
    data={
        "grant_type":    "authorization_code",
        "client_id":     CLIENT_ID,
        "client_secret": CLIENT_SECRET,
        "redirect_uri":  REDIRECT_URI,
        "code":          AUTH_CODE,
    },
)
print(f"Status: {resp.status_code}")
print(f"Body:   {resp.text}")
resp.raise_for_status()
tokens = resp.json()

ACCESS_TOKEN  = tokens["access_token"]
REFRESH_TOKEN = tokens.get("refresh_token")
print(f"access_token: {ACCESS_TOKEN[:20]}...")
print(f"expires_in:   {tokens.get('expires_in')}s")


### Refresh tokens (run if access token expires)

In [21]:
def refresh_access_token(refresh_token: str) -> str:
    resp = httpx.post(
        f"{AUTH_BASE}/connect/token",
        data={
            "grant_type":    "refresh_token",
            "client_id":     CLIENT_ID,
            "client_secret": CLIENT_SECRET,
            "refresh_token": refresh_token,
        },
    )
    resp.raise_for_status()
    data = resp.json()
    return data["access_token"]

ACCESS_TOKEN = refresh_access_token(REFRESH_TOKEN)

## Accounts

In [ ]:
import polars as pl

headers = {"Authorization": f"Bearer {ACCESS_TOKEN}"}

resp = httpx.get(f"{API_BASE}/data/v1/accounts", headers=headers)
resp.raise_for_status()
accounts = resp.json()["results"]

df_accounts = pl.DataFrame(accounts)
print(f"{len(accounts)} account(s)")
df_accounts

## Balances

In [ ]:
balance_rows = []
for acct in accounts:
    acct_id = acct["account_id"]
    resp = httpx.get(f"{API_BASE}/data/v1/accounts/{acct_id}/balance", headers=headers)
    resp.raise_for_status()
    for b in resp.json()["results"]:
        b["account_id"] = acct_id
        b["display_name"] = acct.get("display_name", "")
        balance_rows.append(b)

df_balances = pl.DataFrame(balance_rows)
df_balances

## Transactions

In [ ]:
from datetime import date, timedelta

# fetch the last 90 days for the first account; adjust as needed
acct_id   = accounts[0]["account_id"]
date_from = (date.today() - timedelta(days=90)).isoformat()
date_to   = date.today().isoformat()

resp = httpx.get(
    f"{API_BASE}/data/v1/accounts/{acct_id}/transactions",
    headers=headers,
    params={"from": date_from, "to": date_to},
    timeout=30,
)
resp.raise_for_status()
transactions = resp.json()["results"]

print(f"{len(transactions)} transaction(s) from {date_from} to {date_to}")
transactions[:2]

In [36]:
df_txn = (
    pl.DataFrame(transactions)
    .select(["timestamp", "description", "amount", "currency", "transaction_type", "transaction_category"])
    .with_columns(pl.col("timestamp").str.to_datetime(time_zone='UTC'))
    .sort("timestamp", descending=True)
)
df_txn

timestamp,description,amount,currency,transaction_type,transaction_category
"datetime[μs, UTC]",str,f64,str,str,str
2026-05-08 00:00:00 UTC,"""trainline +443332022222""",-8.8,"""GBP""","""DEBIT""","""PURCHASE"""
2026-05-08 00:00:00 UTC,"""STAGECOACH SERVICESTOCKPORT""",-2.0,"""GBP""","""DEBIT""","""PURCHASE"""
2026-05-08 00:00:00 UTC,"""WH Smith Bolton Bolton""",-4.68,"""GBP""","""DEBIT""","""PURCHASE"""
2026-05-08 00:00:00 UTC,"""Shajan J Adavanal Dad""",-1273.0,"""GBP""","""DEBIT""","""BILL_PAYMENT"""
2026-05-08 00:00:00 UTC,"""Babu S A Gisha holiday""",-2000.0,"""GBP""","""DEBIT""","""BILL_PAYMENT"""
…,…,…,…,…,…
2026-02-09 00:00:00 UTC,"""WM MORRISONS STOREBOLTON""",-13.1,"""GBP""","""DEBIT""","""PURCHASE"""
2026-02-09 00:00:00 UTC,"""BEENETWORK.COM/CHA0161 244 100…",-4.3,"""GBP""","""DEBIT""","""PURCHASE"""
2026-02-08 00:00:00 UTC,"""InvestEngine UK IT51328716""",-2000.0,"""GBP""","""DEBIT""","""DEBIT"""


In [40]:
# Monthly spend summary (credits vs debits)
(
    df_txn
    .with_columns(pl.col("timestamp").dt.truncate("1mo").alias("month"))
    .group_by(["month", "transaction_type"])
    .agg(pl.col("amount").sum().alias("total"))
    .sort(["month", "transaction_type"], descending=[True, False])
)

month,transaction_type,total
"datetime[μs, UTC]",str,f64
2026-05-01 00:00:00 UTC,"""CREDIT""",3049.99
2026-05-01 00:00:00 UTC,"""DEBIT""",-5792.88
2026-04-01 00:00:00 UTC,"""CREDIT""",13799.05
2026-04-01 00:00:00 UTC,"""DEBIT""",-11041.26
2026-03-01 00:00:00 UTC,"""CREDIT""",2453.57
2026-03-01 00:00:00 UTC,"""DEBIT""",-4286.4
2026-02-01 00:00:00 UTC,"""CREDIT""",4953.57
2026-02-01 00:00:00 UTC,"""DEBIT""",-9488.01


In [38]:
# Spend by category
(
    df_txn
    .filter(pl.col("transaction_type") == "DEBIT")
    .group_by("transaction_category")
    .agg(pl.col("amount").sum().abs().alias("total"))
    .sort("total", descending=True)
)

transaction_category,total
str,f64
"""DEBIT""",22304.23
"""BILL_PAYMENT""",5666.0
"""PURCHASE""",1837.38
"""DIRECT_DEBIT""",645.94
"""ATM""",130.0
"""STANDING_ORDER""",25.0
